# **Titan Pre-Processing Example**

---

<img align="right" width="300" height="300" src="../../../images/hail_case_tracks.png">

This interactive tutorial takes you through the steps of how to preprocess data in order to run the Thunderstorm Identification, Tracking, Analysis and Nowcasting (Titan) suite. 
The key preprocessing steps for Titan include: 
1) removing noise and non-meteorological echoes (and correcting any biases), and
2) converting from polar to Cartesian grids in the NCAR MDV format. 

This notebook provides a few examples using LROSE applications, but feel free to use your favorite package to pre-process the data. 

Titan was originally designed as an algorithm to objectively identify and track thunderstorms from weather radar data for a weather modification experiment in South Africa in the 1980s. Now, Titan includes forecasting, storm analysis, and climatological analysis. Titan now refers to the larger system in which the original application is one component.

Titan is described in more detail in [Dixon and Wiener (1993)](https://doi.org/10.1175/1520-0426(1993)010%3C0785:TTITAA%3E2.0.CO;2).


---

## **Titan Background**

Titan identifies storm objects as a contiguous region of echo that exceeds a user-defined reflectivity threshold and minimum volume. Dual thresholds are used to deal with storm objects that briefly touch, but do not merge. Storm tracking is performed by looking for regions of overlap between storm objects at successive time intervals. Short term storm extrapolation forecasts are used to identify instances of storm merging and splitting. Titan output includes storm tracks, polygons outlining the storm objects, and storm property information (e.g., volume, area, mass, precipitation flux).

The high-level workflow for Titan is shown in the graphic below. Key steps include quality controlling the data to remove any non-meteorological or compromised echoes and gridding the data to a Cartesian grid. Once Titan is run and the tracks are produced, those data need to be converted into more user-friendly file types. 

<img align="center" width="600" src="../../../images/titan_highlevel.png">

A more detailed workflow for Titan that includes each step, application, and data type is shown in the graphic below.

<img align="center" width="800" src="../../../images/titan_data_flow.png">

## **Tutorial Overview**
### 1. Setup

#### Raw data and parameter files

Files required to run this notebook:
* Raw radar data between 1600-1700 UTC.

Jastrebac is a 10 cm (S-band) Gematronik dual polarization radar in Serbia, supervised by the Republic Hydrometeorological Service of Serbia (RHMZ). This hourlong period contains areas of convection in the region.

We've provided the raw Rainbow formatted data.

### 2. Output data

After the full analysis is run, the following data files should exist:

```
  ./data/titan_preprocess/convert/cfrad (TITAN binary files)
  ./data/titan_preprocess/convert_qc/ (
  ./data/titan_preprocess/grid
```

### 3. Note on task cells

This notebook uses two colored cells to indicate tasks.

<div class="alert alert-block alert-info"> <b>File Task: modify parameters in text files.</b> 

These text blocks help the user modify the parameter files or other functions in *external* text files.

</div>

<div class="alert alert-block alert-warning"> <b>Cell Task: run a command in Jupyter notebook cell.</b> 

These text blocks instruct the users to run a command *in* a cell within the Jupyter notebook. If you prefer, you are welcome to copy the commands (minus the ! symbol) into a terminal window.

</div>

---

## **1. Setup**
### Environment and packages

First, we import the required python packages to run this notebook. Most of the LROSE processing can be done with the os package and shell commands.

In [ ]:
import os
import warnings
import pyart
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

### 1.1 Set up directories

We need to set up the required data directories. The raw radar data will be grabbed from the S3 bucket.

In [ ]:
# make overall titan directory and application output directory
!mkdir -p ./data/titan_preprocess


### 1.2 Set up the environment

First, we'll set some key variables we'll need throughout the workflow.

In [ ]:
# Set directory variable to call LROSE
os.environ["LROSE_DIR"] = "/usr/local/lrose/bin"
os.environ['RAW_DIR'] = '/home/jovyan/share/raw'

## **2. Prepare data for analysis**

The following sections describe two quality control setups and how to run the scripts.

First, let's just do a simple format conversion to see what the raw data look like. We can run RadxConvert with its default parameters and inspect the data. 

<div class="alert alert-block alert-warning"> <b>Cell Task: Run default RadxConvert on Serbian data.</b> 
    <br>
    Let's save the output data to a folder called "convert" using the -outdir flag.
    <br>
    <code lang="bash">!$LROSE_DIR/RadxConvert -sort_rays_by_time -const_ngates -f /home/jovyan/share/raw/titan_2026/2017081216*.vol -outdir ./data/titan_preprocess/convert</code>
    
</div>


In [ ]:
ls $RAW_DIR/titan_2026/2017081216*.vol

In [ ]:
!$LROSE_DIR/RadxConvert -sort_rays_by_time -const_ngates -f /home/jovyan/share/raw/titan_2026/*.vol -outdir ./data/titan_preprocess/convert


Now let's use Py-ART to read in the file and look at the variables in a file to assess the data quality.

In [ ]:
radar = pyart.io.read_cfradial("./data/titan_preprocess/convert/20170812/cfrad.20170812_164007.000_to_20170812_164332.958_Jastrebac_SUR.nc")
print(radar.info())


We'll use Py-ART to inspect the quality of reflectivity, velocity, differential reflectivity, and correlation coefficient. 

Note that in these files, the names of the variables are as follows: ***dBZ, V, ZDR, and RhoHV***.

In [ ]:
display = pyart.graph.RadarDisplay(radar)

fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(8, 6), sharex=True, sharey=True)

# --- Panel 1: Reflectivity ---
ax1 = axs[0, 0]
display.plot("dBZ", sweep=0, ax=ax1, title="Reflectivity (dBZ)",
    cmap="ChaseSpectral", vmin=-20, vmax=64)
display.set_limits((-150, 150), (-150, 150), ax=ax1)

# --- Panel 2: Velocity ---
ax2 = axs[0, 1]
display.plot("V", sweep=1, ax=ax2, title="Doppler Velocity (m/s)",
    cmap="balance", vmin=-35, vmax=35)
display.set_limits((-150, 150), (-150, 150), ax=ax2)

# --- Panel 3: Differential Reflectivity ---
ax3 = axs[1, 0]
display.plot("ZDR", sweep=0, ax=ax3, title="Differential Reflectivity (ZDR)",
    cmap="ChaseSpectral", vmin=-2, vmax=6)
display.set_limits((-150, 150), (-150, 150), ax=ax3)

# --- Panel 4: Cross Correlation Ratio ---
ax4 = axs[1, 1]
display.plot("RhoHV", sweep=0, ax=ax4, title="Correlation Coefficient (RhoHV)",
    cmap="ChaseSpectral", vmin=0.5, vmax=1.05)
display.set_limits((-150, 150), (-150, 150), ax=ax4)

plt.tight_layout()
plt.show()

#### Figure analysis

From this plot, we notice a few things. First, the data are fairly clean, with some QC already applied, but there are some regions associated with low RhoHV. Second, the ZDR field is running quite hot, about 4 dB above what we expect.

We can do a simple QC to correct some of these errors.

### 2.1: Option 1 - Apply quality control (QC) on the raw radar data and convert to CfRadial format using RadxConvert

Using a provided parameter file, we will correct the ZDR bias and remove data with low RhoHV values. We will do this using simple QC procedures available in RadxConvert. 

We will apply a -4 dB bias correction to ZDR using the "apply_linear_transforms" parameter and remove data where RhoHV < 0.6 using the "apply_censoring" parameter.

To inspect the parameter file, it can be found here: ./params/RadxConvert.QC

***Note:*** *we've changed the output directory to ./data/titan_preprocess/convert_qc using the -outdir flag so that we retain the data from the previous step.*

<div class="alert alert-block alert-warning"> <b>Cell Task: Run simple rules-based QC on the raw data.</b> 
    <br>
    Quality control and convert the data.
    <br>
    <code lang="bash">!$LROSE_DIR/RadxConvert -sort_rays_by_time -const_ngates -params ./params/RadxConvert.QC -f /home/jovyan/share/raw/titan_2026/*.vol -outdir ./data/titan_preprocess/convert_qc</code>
</div>

In [ ]:
# Run QC 
!$LROSE_DIR/RadxConvert -sort_rays_by_time -const_ngates -params ./params/RadxConvert.QC -f /home/jovyan/share/raw/titan_2026/*.vol -outdir ./data/titan_preprocess/convert_qc


Let's visualize the data with Py-ART again.

In [ ]:
radar_qc = pyart.io.read_cfradial("./data/titan_preprocess/convert_qc/20170812/cfrad.20170812_164007.000_to_20170812_164332.958_Jastrebac_SUR.nc")

display_qc = pyart.graph.RadarDisplay(radar_qc)

fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(8, 6), sharex=True, sharey=True)

# --- Panel 1: Reflectivity ---
ax1 = axs[0, 0]
display_qc.plot("dBZ", sweep=0, ax=ax1, title="Reflectivity (dBZ)",
    cmap="ChaseSpectral", vmin=-20, vmax=64)
display_qc.set_limits((-150, 150), (-150, 150), ax=ax1)

# --- Panel 2: Velocity ---
ax2 = axs[0, 1]
display_qc.plot("V", sweep=1, ax=ax2, title="Doppler Velocity (m/s)",
    cmap="balance", vmin=-35, vmax=35)
display_qc.set_limits((-150, 150), (-150, 150), ax=ax2)

# --- Panel 3: Differential Reflectivity ---
ax3 = axs[1, 0]
display_qc.plot("ZDR", sweep=0, ax=ax3, title="Differential Reflectivity (ZDR)",
    cmap="ChaseSpectral", vmin=-2, vmax=6)
display_qc.set_limits((-150, 150), (-150, 150), ax=ax3)

# --- Panel 4: Cross Correlation Ratio ---
ax4 = axs[1, 1]
display_qc.plot("RhoHV", sweep=0, ax=ax4, title="Correlation Coefficient (RhoHV)",
    cmap="ChaseSpectral", vmin=0.5, vmax=1.05)
display_qc.set_limits((-150, 150), (-150, 150), ax=ax4)

plt.tight_layout()
plt.show()

#### Comparison with the original figure

Now we see that the ZDR values are more reasonable and the data associated with the low RhoHV values have been removed.



### 2.2: Option 2 - Computing PID as an alternative method of censoring using RadxPid

An alternative method for cleaning up interference is to run RadxPid, and censor non-meteorological echoes.



### 2.3: Convert to Cartesian grid using Radx2Grid

Titan requires input data on a Cartesian grid, instead of the native polar grid. To perform this transformation, we run Radx2Grid to convert the data to Cartesian grid MDV format.

<div class="alert alert-block alert-warning"> <b>Cell Task: Convert the QC'd derecho case data to a Cartesian grid.</b> 
    <br>
    <br>
    Run Radx2Grid to interpolate the data to a Cartesian grid:
    <br>
    <br>
    <code lang="bash">!$LROSE_DIR/Radx2Grid -f ./data/titan_preprocess/convert_qc/cfrad*.nc -params ./params/Radx2Grid.params -outdir ./data/titan_preprocess/grid_qc</code>
</div>

## **3: Now you're ready to run Titan!**

Use the Titan_tutorial notebook to learn more about setting up the parameter file, running Titan, and testing out a few key variables.